# Architecture Sensitivity Analysis - Google Colab

Random Forest-based feature importance for Transformer architecture parameters.

In [ ]:
import os
from google.colab import files

repo_url = "https://github.com/bp571/BA"
repo_name = "BA"

if not os.path.exists(repo_name):
    !git clone {repo_url} {repo_name}

%cd {repo_name}

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pandas numpy matplotlib seaborn scikit-learn tqdm yfinance einops peft transformers huggingface_hub chronos-forecasting

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 1: Random Search

Sample architecture parameters and evaluate models.

In [ ]:
n_samples = 150
seed = 42

!python 03_sensitivity_analysis/architecture_parameters/run_architecture_search.py \
    --n-samples {n_samples} \
    --seed {seed}

## Step 2: Random Forest Analysis

Train Random Forest and extract feature importances.

In [ ]:
!python 03_sensitivity_analysis/architecture_parameters/analyze_rf.py

## Step 3: Display Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f"03_sensitivity_analysis/architecture_parameters/results/architecture_search_{n_samples}.csv")
print(f"Results: {len(df)} samples\n")
print(df.describe())
print("\nSample rows:")
display(df.head(10))

In [ ]:
print("Feature Importance - MAE:")
display(Image("03_sensitivity_analysis/architecture_parameters/results/rf_importance_mae.png"))

print("\nFeature Importance - RankIC:")
display(Image("03_sensitivity_analysis/architecture_parameters/results/rf_importance_rankic.png"))

## Step 4: Download Results

In [ ]:
from google.colab import files
import zipfile

zip_name = f"architecture_sensitivity_results.zip"

with zipfile.ZipFile(zip_name, 'w') as zipf:
    zipf.write(f"03_sensitivity_analysis/architecture_parameters/results/architecture_search_{n_samples}.csv",
               arcname=f"architecture_search_{n_samples}.csv")
    zipf.write("03_sensitivity_analysis/architecture_parameters/results/rf_importance_mae.png",
               arcname="rf_importance_mae.png")
    zipf.write("03_sensitivity_analysis/architecture_parameters/results/rf_importance_rankic.png",
               arcname="rf_importance_rankic.png")

files.download(zip_name)
print(f"Downloaded: {zip_name}")